# Anova and Tukey HSD Tests
## One way or two way ANOVA specified in formula string

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.anova import anova_lm
from scipy.stats import levene  # Corrected import
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# ---------------------------
# Step 1: Define File Paths
# ---------------------------

# Path to the aggregated bootstrapped results CSV
csv_path = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\aggregated_bootstrap_results.csv"

# Two-way ANOVA formula with interactions
# anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors_Interaction"
# formula = 'Coefficient_Estimate ~ C(Fire_Type) + C(Predictor) + C(Fire_Type):C(Predictor)'

# One way ANOVA formula
# anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\One-way_ANOVA"
# formula = 'Coefficient_Estimate ~ C(Fire_Type)'

# Two way ANOVA formula
anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors"
formula = 'Coefficient_Estimate ~ C(Fire_Type) + C(Predictor)'

# Create subdirectories
os.makedirs(anova_results_dir, exist_ok=True)
os.makedirs(os.path.join(anova_results_dir, "Individual Cases"), exist_ok=True)

# ---------------------------
# Step 2: Load and Prepare Data
# ---------------------------

# Load the data from the CSV file
try:
    df = pd.read_csv(csv_path)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file at '{csv_path}' was not found.")
    exit()

# Select relevant columns
required_columns = ['Region', 'term', 'estimate', 'bootstrap_rep', 'Segmentation Interval', 'Dependent Variable', 'p.value']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"Error: The following required columns are missing in the CSV: {missing_columns}")
    exit()

anova_df = df[required_columns].copy()

# Remove rows with p.value greater than 0.05
anova_df = anova_df[anova_df['p.value'] <= 0.05]

# Rename columns for clarity
anova_df = anova_df.rename(columns={
    'Region': 'Fire_Type',
    'term': 'Predictor',
    'estimate': 'Coefficient_Estimate',
    'Segmentation Interval': 'Segmentation_Interval',
    'Dependent Variable': 'Dependent_Variable'
})

# Convert categorical variables to 'category' dtype
anova_df['Fire_Type'] = anova_df['Fire_Type'].astype('category')
anova_df['Predictor'] = anova_df['Predictor'].astype('category')
anova_df['Segmentation_Interval'] = anova_df['Segmentation_Interval'].astype('category')
anova_df['Dependent_Variable'] = anova_df['Dependent_Variable'].astype('category')

# Remove the '(Intercept)' Predictor to avoid multicollinearity
anova_df = anova_df[anova_df['Predictor'] != '(Intercept)']

# ---------------------------
# Step 3: Identify and Remove Problematic Predictors
# ---------------------------

# a. Identify Predictors with Only One Fire Type
single_fire_predictors = anova_df.groupby('Predictor')['Fire_Type'].nunique()
single_fire_predictors = single_fire_predictors[single_fire_predictors < 2].index.tolist()

print("\nPredictors with only one Fire Type:")
print(single_fire_predictors)

# b. Remove Predictors with Only One Fire Type
anova_df = anova_df[~anova_df['Predictor'].isin(single_fire_predictors)]
print(f"\nRemoved {len(single_fire_predictors)} predictors with single Fire Type.")

# c. Identify Predictors with Zero Variance
zero_variance_predictors = anova_df.groupby('Predictor')['Coefficient_Estimate'].var()
zero_variance_predictors = zero_variance_predictors[zero_variance_predictors == 0].index.tolist()

print("\nPredictors with zero variance in Coefficient Estimates:")
print(zero_variance_predictors)

# d. Remove Predictors with Zero Variance
anova_df = anova_df[~anova_df['Predictor'].isin(zero_variance_predictors)]
print(f"\nRemoved {len(zero_variance_predictors)} predictors with zero variance.")

# Save the cleaned dataframe
cleaned_csv_path = os.path.join(anova_results_dir, "cleaned_bootstrap_results.csv")
anova_df.to_csv(cleaned_csv_path, index=False)

# ---------------------------
# Step 4: Define Functions
# ---------------------------

def perform_anova(subset, segmentation, dependent_var, formula):
    """
    Performs Two-Way ANOVA on a filtered subset of data.

    Parameters:
    - subset (DataFrame): Data filtered for a specific Segmentation Interval and Dependent Variable.
    - segmentation (str): Identifier for the current Segmentation Interval.
    - dependent_var (str): Identifier for the current Dependent Variable.
    - formula (str): Model formula that defines the relationship between the response variable (Coefficient_Estimate)
                     and the categorical predictors (e.g., Fire_Type, Predictor) as well as their interaction if needed.

    Returns:
    - anova_table (DataFrame): DataFrame containing the ANOVA results (sum of squares, degrees of freedom, F-statistic, and p-value).
    - model (RegressionResultsWrapper): The fitted Ordinary Least Squares (OLS) model object for further diagnostics if required.
    """
    try:
        # Fit the OLS regression model using the specified formula and subset.
        # The formula typically includes the effects of both categorical variables and their interaction.
        model = ols(formula, data=subset).fit()

        # ANOVA statsmodels sums of squares Types:

        # - Type I (Sequential SS):
        #   * Computes the sum of squares sequentially, assigning variance to each factor in the order
        #     they are entered in the model.
        #   * The test is order-dependent, meaning the result for a given factor can change if the order
        #     of predictors is altered.
        #
        # - Type II (Balanced SS):
        #   * Tests each main effect after accounting for all other main effects (ignoring interactions).
        #   * It is less sensitive to the ordering of variables and is suitable for unbalanced designs 
        #     (unequal sample sizes among groups).
        #   * This method is often preferred when interactions are not the primary focus of the analysis.
        #
        # - Type III (Marginal SS):
        #   * Tests each effect (both main effects and interactions) after adjusting for all other effects in the model.
        #   * It is commonly used in factorial designs and is the default in many statistical software packages.
        #   * This method is particularly useful when interactions are present and need to be accounted for.
        #
        anova_table = sm.stats.anova_lm(model, typ=2)

        # Return both the ANOVA table and the fitted model.
        return anova_table, model

    except Exception as e:
        # Log an error message with contextual information if model fitting or ANOVA fails.
        print(f"Error performing Two-Way ANOVA for '{segmentation}' / '{dependent_var}': {e}")
        return None, None


def perform_tukey_hsd_within_predictor(subset, predictor, segmentation, dependent_var):
    """
    Performs Tukey's Honest Significant Difference (HSD) test to compare Coefficient_Estimate values 
    between different Fire Types within a single Predictor group.

    Parameters:
    - subset (DataFrame): Data filtered for a specific Segmentation Interval and Dependent Variable.
    - predictor (str): The Predictor for which the Tukey test is to be applied.
    - segmentation (str): Identifier for the Segmentation Interval.
    - dependent_var (str): Identifier for the Dependent Variable.

    Returns:
    - tukey_df (DataFrame): DataFrame containing Tukey's HSD results, including group comparisons, 
                            mean differences, confidence intervals, and p-values.
      Returns None if the conditions for performing Tukey's HSD are not met.
    """
    # Filter the data to include only rows for the given predictor.
    predictor_subset = subset[subset['Predictor'] == predictor]

    # Retrieve the unique Fire Types present in the predictor subset.
    # Tukey's HSD requires a pairwise comparison; thus, exactly two groups are expected.
    fire_types = predictor_subset['Fire_Type'].unique()
    if len(fire_types) != 2:
        # If the number of groups is not equal to 2, log a message and skip Tukey's HSD for this predictor.
        print(f"Skipping Tukey's HSD for Predictor '{predictor}' - Expected 2 Fire Types, found {len(fire_types)}")
        return None

    try:
        # Perform Tukey's HSD test.
        # - endog: the dependent variable (Coefficient_Estimate).
        # - groups: the categorical variable (Fire_Type) to define the groups for comparison.
        # - alpha: significance level for the test.
        tukey = pairwise_tukeyhsd(endog=predictor_subset['Coefficient_Estimate'],
                                  groups=predictor_subset['Fire_Type'],
                                  alpha=0.05)

        # Convert the Tukey test results (which are returned as a summary table) into a DataFrame.
        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])

        # Append additional columns 
        tukey_df['Segmentation_Interval'] = segmentation
        tukey_df['Dependent_Variable'] = dependent_var
        tukey_df['Predictor'] = predictor

        # Return the formatted DataFrame containing Tukey's HSD results.
        return tukey_df

    except Exception as e:
        # Log an error message if Tukey's HSD fails for the given predictor.
        print(f"Error performing Tukey's HSD for Predictor '{predictor}': {e}")
        return None

# ---------------------------
# Step 5: Perform Two-Way ANOVA and Tukey's HSD
# ---------------------------

# Get unique combinations of Segmentation Interval and Dependent Variable
combinations = anova_df[['Segmentation_Interval', 'Dependent_Variable']].drop_duplicates()

# Initialize lists to collect master results
master_anova_results = []
master_tukey_results = []

anova_name = os.path.basename(anova_results_dir)

# Iterate through each combination
for index, row in combinations.iterrows():
    segmentation = row['Segmentation_Interval']
    dependent_var = row['Dependent_Variable']
    
    print(f"\nPerforming {anova_name} for Segmentation Interval: '{segmentation}' and Dependent Variable: '{dependent_var}'")
    
    # Subset the data for the current combination
    subset = anova_df[
        (anova_df['Segmentation_Interval'] == segmentation) &
        (anova_df['Dependent_Variable'] == dependent_var)
    ]
    
    # Data sufficiency checks
    fire_types = subset['Fire_Type'].unique()
    print(f"Fire Types: {fire_types}")
    if len(fire_types) < 2:
        print(f"Skipping ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Regiom Types.")
        continue
    
    predictors = subset['Predictor'].unique()
    if len(predictors) < 2:
        print(f"Skipping  ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Predictors.")
        continue
    
    bootstrap_reps = subset['bootstrap_rep'].unique()
    if len(bootstrap_reps) < 2:
        print(f"Skipping  ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Bootstrap Replications.")
        continue
    
    # Perform ANOVA
    anova_results = perform_anova(subset, segmentation, dependent_var, formula)
    if anova_results is None:
        print(f"Skipping further analysis for '{segmentation}' / '{dependent_var}' due to ANOVA errors.")
        continue
    
    anova_table, model = anova_results
    print("\nANOVA Table:")
    print(anova_table)
    
    # Save ANOVA table to CSV
    anova_filename = os.path.join(anova_results_dir, "Individual Cases", f"ANOVA_{segmentation}_{dependent_var}.csv")
    anova_table.to_csv(anova_filename)
    print(f"Two-Way ANOVA results saved to '{anova_filename}'")
    
    # Optional: Calculate and store effect sizes (e.g., Eta Squared)
    anova_table = anova_table.reset_index()
    anova_table['Segmentation_Interval'] = segmentation
    anova_table['Dependent_Variable'] = dependent_var
    anova_table['Eta_Squared'] = anova_table['sum_sq'] / anova_table['sum_sq'].sum()
    master_anova_results.append(anova_table)
    
    # Since the model does not include interaction, skip interaction checks
    print("No interaction term in the model. Tukey's HSD test not performed.")
    
    # If you still want to perform Tukey's HSD without interaction:
    print("Performing Tukey's HSD test within each Predictor...")
    tukey_results = []
    for predictor in predictors:
        tukey_df = perform_tukey_hsd_within_predictor(subset, predictor, segmentation, dependent_var)
        if tukey_df is not None:
            tukey_results.append(tukey_df)
    
    if tukey_results:
        master_tukey_df = pd.concat(tukey_results, ignore_index=True)
        master_tukey_results.append(master_tukey_df)
        tukey_filename = os.path.join(anova_results_dir, "Individual Cases", f"Tukey_HSD_{segmentation}_{dependent_var}.csv")
        master_tukey_df.to_csv(tukey_filename, index=False)
        print(f"Tukey's HSD results saved to '{tukey_filename}'")
    else:
        print("No Tukey's HSD results to save.")
        
    
    # ---------------------------
    # Step 5.5: Validate Assumptions
    # ---------------------------
    from scipy.stats import shapiro

    residuals = model.resid
    stat, p = shapiro(residuals)
    if p < 0.05:
        print("\nWarning: Residuals deviate from normality (p-value = {:.3f}).".format(p))
    else:
        print("\nResiduals appear normally distributed (p-value = {:.3f}).".format(p))

    import matplotlib.pyplot as plt
    import seaborn as sns
    import statsmodels.api as sm
    import numpy as np


    # 1. Residuals vs Fitted Values Plot
    plt.figure(figsize=(8, 6))
    sns.residplot(x=model.fittedvalues, y=model.resid, lowess=True,
                line_kws={'color': 'red', 'lw': 1})
    plt.xlabel('Fitted Values')
    plt.ylabel('Residuals')
    plt.title('Residuals vs Fitted Values')
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(anova_results_dir, "Individual Cases", f"Residuals_vs_Fitted_{segmentation}_{dependent_var}.png"), dpi=300)

    # 2. Normal Q-Q Plot of Residuals
    fig = sm.qqplot(model.resid, line='45', fit=True)
    plt.title('Normal Q-Q Plot')
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(anova_results_dir, "Individual Cases", f"Normal_QQ_{segmentation}_{dependent_var}.png"), dpi=300)

    # 3. Histogram of Residuals with KDE
    plt.figure(figsize=(8, 6))
    sns.histplot(model.resid, kde=True)
    plt.xlabel('Residuals')
    plt.title('Histogram of Residuals')
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(anova_results_dir, "Individual Cases", f"Histogram_Residuals_{segmentation}_{dependent_var}.png"), dpi=300)

# ---------------------------
# Step 6: Consolidate ANOVA and Tukey's HSD Results
# ---------------------------

# Consolidate ANOVA results into a master CSV
if master_anova_results:
    master_anova_df = pd.concat(master_anova_results, ignore_index=True)
    
    # Save master ANOVA results
    master_anova_filename = os.path.join(anova_results_dir, "Aggregated_ANOVA_Results.csv")
    master_anova_df.to_csv(master_anova_filename, index=False)
    print(f"\nMaster ANOVA results saved to '{master_anova_filename}'")
else:
    print("\nNo ANOVA results to consolidate.")

# Consolidate Tukey's HSD results into a master CSV
if master_tukey_results:
    master_tukey_df = pd.concat(master_tukey_results, ignore_index=True)
    master_tukey_filename = os.path.join(anova_results_dir, "Aggregated_Tukey_HSD_Results.csv")
    master_tukey_df.to_csv(master_tukey_filename, index=False)
    print(f"Master Tukey's HSD results saved to '{master_tukey_filename}'")
else:
    print("\nNo Tukey's HSD results to consolidate.")

# ---------------------------
# Step 7: Visualize and Save Aggregated Results (with Legend for Tukey HSD)
# ---------------------------

# Visualize Aggregated ANOVA Results (Effect Sizes)
if master_anova_results:
    # Rename the 'index' column to 'Factor' if needed
    if 'index' in master_anova_df.columns:
        master_anova_df = master_anova_df.rename(columns={'index': 'Factor'})
    
    # Create a facet grid bar plot of Eta Squared for each factor, faceted by Segmentation Interval and Dependent Variable.
    sns.set_theme(style="whitegrid")
    g = sns.catplot(
        data=master_anova_df,
        x='Factor',
        y='Eta_Squared',
        hue='Factor',
        col='Segmentation_Interval',
        row='Dependent_Variable',
        kind='bar',
        height=4,
        aspect=1.5,
        legend=False
    )
    g.set_xticklabels(rotation=45)
    g.set_titles(row_template="{row_name}", col_template="{col_name}")
    plt.tight_layout()
    
    anova_plot_filename = os.path.join(anova_results_dir, "Aggregated_ANOVA_EtaSquared.png")
    plt.savefig(anova_plot_filename, dpi=300)
    plt.close()
    print(f"Aggregated ANOVA Eta Squared plot saved to '{anova_plot_filename}'")
else:
    print("No aggregated ANOVA results available for plotting.")

# Visualize Aggregated Tukey HSD Results (Mean Differences with Confidence Intervals and Legend)
if master_tukey_results:
    for seg in master_tukey_df['Segmentation_Interval'].unique():
        for dep in master_tukey_df['Dependent_Variable'].unique():
            df_subset = master_tukey_df[
                (master_tukey_df['Segmentation_Interval'] == seg) & 
                (master_tukey_df['Dependent_Variable'] == dep)
            ]
            if df_subset.empty:
                continue

            plt.figure(figsize=(8, 6))
            
            # Reset the index and assign an x-position to each row for plotting
            df_subset = df_subset.copy().reset_index(drop=True)
            df_subset['x_position'] = np.arange(len(df_subset))
            
            # Initialize a set to keep track of labels already added to avoid duplicate legend entries
            labels_added = set()
            
            # Group by the 'reject' column and plot error bars with different colors/labels
            for reject_value, group in df_subset.groupby('reject'):
                # Convert the reject value to a string and compare to 'TRUE'
                if str(reject_value).upper() == 'TRUE':
                    color = 'red'
                    label = 'Significant difference'
                else:
                    color = 'blue'
                    label = 'Non-significant difference'
                # Only assign the label once per group to avoid duplicate legend entries
                label_to_use = label if label not in labels_added else None
                labels_added.add(label)
                
                x = group['x_position']
                y = group['meandiff']
                lower_err = y - group['lower']
                upper_err = group['upper'] - y
                plt.errorbar(
                    x,
                    y,
                    yerr=[lower_err, upper_err],
                    fmt='o',
                    capsize=5,
                    color=color,
                    label=label_to_use
                )
            
            plt.xticks(df_subset['x_position'], df_subset['Predictor'], rotation=45)
            plt.xlabel("Predictor")
            plt.ylabel("Mean Difference")
            plt.title(f"Tukey HSD Mean Differences\nSegmentation: {seg}, Dependent Variable: {dep}")
            plt.legend()
            plt.tight_layout()
            
            tukey_plot_filename = os.path.join(anova_results_dir, f"Aggregated_Tukey_HSD_{seg}_{dep}.png")
            plt.savefig(tukey_plot_filename, dpi=300)
            plt.close()
            print(f"Aggregated Tukey HSD plot saved to '{tukey_plot_filename}'")
else:
    print("No aggregated Tukey HSD results available for plotting.")






In [14]:
# ---------------------------
# Step 7: Visualize and Save Aggregated Results (with Legend for Tukey HSD)
# ---------------------------

# Visualize Aggregated ANOVA Results (Effect Sizes)
if master_anova_results:
    # Rename the 'index' column to 'Factor' if needed
    if 'index' in master_anova_df.columns:
        master_anova_df = master_anova_df.rename(columns={'index': 'Factor'})
    
    # Create a facet grid bar plot of Eta Squared for each factor, faceted by Segmentation Interval and Dependent Variable.
    sns.set_theme(style="whitegrid")
    g = sns.catplot(
        data=master_anova_df,
        x='Factor',
        y='Eta_Squared',
        hue='Factor',
        col='Segmentation_Interval',
        row='Dependent_Variable',
        kind='bar',
        height=4,
        aspect=1.5,
        legend=False
    )
    g.set_xticklabels(rotation=45)
    g.set_titles(row_template="{row_name}", col_template="{col_name}")
    plt.tight_layout()
    
    anova_plot_filename = os.path.join(anova_results_dir, "Aggregated_ANOVA_EtaSquared.png")
    plt.savefig(anova_plot_filename, dpi=300)
    plt.close()
    print(f"Aggregated ANOVA Eta Squared plot saved to '{anova_plot_filename}'")
else:
    print("No aggregated ANOVA results available for plotting.")

# Visualize Aggregated Tukey HSD Results (Mean Differences with Confidence Intervals and Legend)
if master_tukey_results:
    for seg in master_tukey_df['Segmentation_Interval'].unique():
        for dep in master_tukey_df['Dependent_Variable'].unique():
            df_subset = master_tukey_df[
                (master_tukey_df['Segmentation_Interval'] == seg) & 
                (master_tukey_df['Dependent_Variable'] == dep)
            ]
            if df_subset.empty:
                continue

            plt.figure(figsize=(8, 6))
            
            # Reset the index and assign an x-position to each row for plotting
            df_subset = df_subset.copy().reset_index(drop=True)
            df_subset['x_position'] = np.arange(len(df_subset))
            
            # Initialize a set to keep track of labels already added to avoid duplicate legend entries
            labels_added = set()
            
            # Group by the 'reject' column and plot error bars with different colors/labels
            for reject_value, group in df_subset.groupby('reject'):
                # Convert the reject value to a string and compare to 'TRUE'
                if str(reject_value).upper() == 'TRUE':
                    color = 'red'
                    label = 'Significant difference'
                else:
                    color = 'blue'
                    label = 'Non-significant difference'
                # Only assign the label once per group to avoid duplicate legend entries
                label_to_use = label if label not in labels_added else None
                labels_added.add(label)
                
                x = group['x_position']
                y = group['meandiff']
                lower_err = y - group['lower']
                upper_err = group['upper'] - y
                plt.errorbar(
                    x,
                    y,
                    yerr=[lower_err, upper_err],
                    fmt='o',
                    capsize=5,
                    color=color,
                    label=label_to_use
                )
            
            plt.xticks(df_subset['x_position'], df_subset['Predictor'], rotation=45)
            plt.xlabel("Predictor")
            plt.ylabel("Mean Difference")
            plt.title(f"Tukey HSD Mean Differences\nSegmentation: {seg}, Dependent Variable: {dep}")
            plt.legend()
            plt.tight_layout()
            
            tukey_plot_filename = os.path.join(anova_results_dir, f"Aggregated_Tukey_HSD_{seg}_{dep}.png")
            plt.savefig(tukey_plot_filename, dpi=300)
            plt.close()
            print(f"Aggregated Tukey HSD plot saved to '{tukey_plot_filename}'")
else:
    print("No aggregated Tukey HSD results available for plotting.")


Aggregated ANOVA Eta Squared plot saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors\Aggregated_ANOVA_EtaSquared.png'
Aggregated Tukey HSD plot saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors\Aggregated_Tukey_HSD_Segmented 10m_lidar_erosion_logtrans.png'
Aggregated Tukey HSD plot saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors\Aggregated_Tukey_HSD_Segmented 10m_sfm_deposition_logtrans.png'
Aggregated Tukey HSD plot saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors\Aggregated_Tukey_HSD_Segmented 10m_sfm_erosion_logtrans.png'
Aggregated Tukey HSD plot saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Predictors\Aggregated_Tukey_HSD_Segmented 10m_sfm_net_logtrans.png'

## Residual checks for bootstrapping models